# P2 Core Matrices and Normalization

This notebook implements **Phase P2** from the frozen pipeline.

## Goals

- construct the country-by-mode count matrix
- construct the country-by-mode capacity matrix
- export raw matrices
- compute row-normalized matrices
- compute projection-based concentration diagnostics
- check whether normalization materially reduces head-country dominance

## Required outputs

- `artifacts/tables/p2_country_mode_count_raw.csv`
- `artifacts/tables/p2_country_mode_capacity_raw.csv`
- `artifacts/tables/p2_country_mode_count_row_normalized.csv`
- `artifacts/tables/p2_country_mode_capacity_row_normalized.csv`
- `artifacts/tables/p2_concentration_diagnostics.csv`
- `artifacts/reports/p2_normalization_report.md`

This notebook uses the frozen `P1` retained baseline as its only analytical input.


In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p1_retained_baseline.csv').exists():
    ROOT = ROOT.parent

INPUT_TABLE = ROOT / 'artifacts' / 'tables' / 'p1_retained_baseline.csv'
TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P2__nogit'
STATUS_ORDER = ['operating', 'construction', 'pre-construction', 'announced']
SIZE_BIN_LABELS = ['1-5', '5-50', '50-500', '500+']
MODE_ORDER = [f'{status} | {label}' for status in STATUS_ORDER for label in SIZE_BIN_LABELS]
EXPECTED_COUNTRIES = 94
EXPECTED_ROWS = 98942
EXPECTED_CAPACITY_MW = 3573038.8
PROJECTION_RULES = {
    'raw': {'similarity': 'dot', 'top_k': 6},
    'row_normalized': {'similarity': 'cosine', 'top_k': 6},
}

manifest = {
    'notebook': 'P2_Core_Matrices_and_Normalization.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'input_table': str(INPUT_TABLE),
    'expected_scope': {
        'countries': EXPECTED_COUNTRIES,
        'rows': EXPECTED_ROWS,
        'capacity_mw': EXPECTED_CAPACITY_MW,
    },
    'projection_rules': PROJECTION_RULES,
}
(RUNTIME / 'p2_normalization_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown(f"**Input table:** `{INPUT_TABLE.name}`"))


## Load the Frozen P1 Retained Baseline

Before building matrices, this notebook verifies that the P1 baseline still matches the frozen retained scope. If the input scope drifts here, P2 should stop rather than silently continuing.


In [ ]:
retained = pd.read_csv(INPUT_TABLE, low_memory=False)
retained['Capacity (MW)'] = pd.to_numeric(retained['Capacity (MW)'], errors='coerce')
retained['Country/Area'] = retained['Country/Area'].astype(str)
retained['Status'] = retained['Status'].astype(str)
retained['mode'] = retained['mode'].astype(str)

scope_check = {
    'countries': int(retained['Country/Area'].nunique()),
    'rows': int(len(retained)),
    'capacity_mw': float(retained['Capacity (MW)'].sum()),
    'modes': int(retained['mode'].nunique()),
}

if scope_check['countries'] != EXPECTED_COUNTRIES:
    raise ValueError(f"P1 baseline drift: expected {EXPECTED_COUNTRIES} countries, got {scope_check['countries']}")
if scope_check['rows'] != EXPECTED_ROWS:
    raise ValueError(f"P1 baseline drift: expected {EXPECTED_ROWS} rows, got {scope_check['rows']}")
if not math.isclose(scope_check['capacity_mw'], EXPECTED_CAPACITY_MW, rel_tol=0, abs_tol=0.1):
    raise ValueError(f"P1 baseline drift: expected {EXPECTED_CAPACITY_MW} MW, got {scope_check['capacity_mw']}")
if scope_check['modes'] != len(MODE_ORDER):
    raise ValueError(f"Mode drift: expected {len(MODE_ORDER)} modes, got {scope_check['modes']}")

display(pd.Series(scope_check, name='value').to_frame())


## Build Raw and Row-Normalized Country-by-Mode Matrices

Raw matrices are exported for diagnostic use only. The row-normalized versions are the frozen inputs for later projection-based community analysis.


In [ ]:
def build_matrix(df, weight_kind):
    if weight_kind == 'count':
        work = df.assign(weight=1)
    elif weight_kind == 'capacity_mw':
        work = df.assign(weight=df['Capacity (MW)'].astype(float))
    else:
        raise ValueError(f'Unknown weight kind: {weight_kind}')

    matrix = work.pivot_table(
        index='Country/Area',
        columns='mode',
        values='weight',
        aggfunc='sum',
        fill_value=0,
    )
    matrix = matrix.reindex(columns=MODE_ORDER, fill_value=0)
    matrix = matrix.reindex(sorted(matrix.index), axis=0)
    return matrix


def row_normalize(matrix):
    row_sums = matrix.sum(axis=1).replace(0, np.nan)
    return matrix.div(row_sums, axis=0).fillna(0)


def top_share(series, n=5):
    ordered = pd.Series(series).sort_values(ascending=False)
    total = ordered.sum()
    if total <= 0:
        return np.nan
    return float(ordered.head(n).sum() / total)


def build_projection_graph(matrix, similarity='cosine', top_k=6):
    names = list(matrix.index)
    values = matrix.to_numpy(dtype=float)
    graph = nx.Graph()
    graph.add_nodes_from(names)

    if len(names) < 2:
        return graph

    if similarity == 'cosine':
        sim = cosine_similarity(values)
    elif similarity == 'dot':
        sim = values @ values.T
    else:
        raise ValueError(f'Unknown similarity: {similarity}')

    np.fill_diagonal(sim, 0)
    k = min(top_k, len(names) - 1)

    for i, source in enumerate(names):
        neighbor_idx = np.argsort(sim[i])[-k:]
        for j in neighbor_idx:
            if i == j:
                continue
            weight = float(sim[i, j])
            if weight <= 0:
                continue
            target = names[j]
            if graph.has_edge(source, target):
                graph[source][target]['weight'] = max(graph[source][target]['weight'], weight)
            else:
                graph.add_edge(source, target, weight=weight)

    return graph


count_raw = build_matrix(retained, 'count').astype(int)
capacity_raw = build_matrix(retained, 'capacity_mw').astype(float).round(1)
count_row = row_normalize(count_raw)
capacity_row = row_normalize(capacity_raw)

count_raw_export = count_raw.reset_index()
capacity_raw_export = capacity_raw.reset_index()
count_row_export = count_row.reset_index()
capacity_row_export = capacity_row.reset_index()

count_raw_export.to_csv(TABLES / 'p2_country_mode_count_raw.csv', index=False)
capacity_raw_export.to_csv(TABLES / 'p2_country_mode_capacity_raw.csv', index=False)
count_row_export.to_csv(TABLES / 'p2_country_mode_count_row_normalized.csv', index=False, float_format='%.12f')
capacity_row_export.to_csv(TABLES / 'p2_country_mode_capacity_row_normalized.csv', index=False, float_format='%.12f')

display(count_raw_export.head(5))
display(capacity_raw_export.head(5))


## Concentration Diagnostics and Go/Stop Check

The frozen P2 go/stop rule is evaluated on the country projection, not on matrix row totals.

- raw matrices use a `dot`-based country projection with `top_k=6`
- row-normalized matrices use a `cosine`-based country projection with `top_k=6`
- the diagnostic compares the top-5 share of weighted country strength before and after normalization


In [ ]:
diagnostic_rows = []
for weight_kind, raw_matrix, row_matrix in [
    ('count', count_raw, count_row),
    ('capacity_mw', capacity_raw, capacity_row),
]:
    for variant, matrix in [('raw', raw_matrix), ('row_normalized', row_matrix)]:
        rule = PROJECTION_RULES[variant]
        projection = build_projection_graph(matrix, similarity=rule['similarity'], top_k=rule['top_k'])
        country_strength = pd.Series(dict(projection.degree(weight='weight')), dtype=float).sort_values(ascending=False)
        row_sums = matrix.sum(axis=1)
        diagnostic_rows.append({
            'weight_kind': weight_kind,
            'matrix_variant': variant,
            'diagnostic_basis': 'country_projection_strength',
            'similarity': rule['similarity'],
            'top_k': int(rule['top_k']),
            'countries': int(matrix.shape[0]),
            'modes': int(matrix.shape[1]),
            'graph_nodes': int(projection.number_of_nodes()),
            'graph_edges': int(projection.number_of_edges()),
            'total_weight': float(matrix.to_numpy().sum()),
            'total_strength': round(float(country_strength.sum()), 6),
            'head_country': country_strength.index[0],
            'head_country_share': round(float(country_strength.iloc[0] / country_strength.sum()), 6),
            'top5_share': round(top_share(country_strength, n=5), 6),
            'row_sum_min': round(float(row_sums.min()), 6),
            'row_sum_max': round(float(row_sums.max()), 6),
        })

concentration_diagnostics = pd.DataFrame(diagnostic_rows)
comparison = (
    concentration_diagnostics.pivot(index='weight_kind', columns='matrix_variant', values='top5_share')
    .reset_index()
    .rename_axis(columns=None)
)
comparison['top5_share_drop'] = (comparison['raw'] - comparison['row_normalized']).round(6)
comparison['passes_relief'] = comparison['row_normalized'] < comparison['raw']
concentration_diagnostics = concentration_diagnostics.merge(comparison, on='weight_kind', how='left')
concentration_diagnostics.to_csv(TABLES / 'p2_concentration_diagnostics.csv', index=False)

if not comparison['passes_relief'].all():
    raise ValueError('P2 stop condition triggered: row normalization did not reduce head-country dominance for all weight kinds.')

row_audit = concentration_diagnostics.set_index(['weight_kind', 'matrix_variant'])[['row_sum_min', 'row_sum_max']]

report_lines = [
    '# P2 Normalization Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    f'- input_table: `{INPUT_TABLE.name}`',
    '',
    '## Input scope check',
    f'- countries: {scope_check["countries"]}',
    f'- rows: {scope_check["rows"]:,}',
    f'- capacity_mw: {scope_check["capacity_mw"]:,.1f}',
    f'- modes: {scope_check["modes"]}',
    '',
    '## Projection rules',
    '- raw: country projection on rows using dot similarity and top_k=6',
    '- row_normalized: country projection on rows using cosine similarity and top_k=6',
    '',
    '## Raw vs row-normalized top-5 share',
]
for row in comparison.itertuples(index=False):
    report_lines.append(
        f'- {row.weight_kind}: raw={row.raw:.6f}, row_normalized={row.row_normalized:.6f}, drop={row.top5_share_drop:.6f}, passes_relief={bool(row.passes_relief)}'
    )
report_lines.extend([
    '',
    '## Matrix row-sum audit',
    f'- count raw row sums: min={row_audit.loc[("count", "raw"), "row_sum_min"]:.6f}, max={row_audit.loc[("count", "raw"), "row_sum_max"]:.6f}',
    f'- count row-normalized row sums: min={row_audit.loc[("count", "row_normalized"), "row_sum_min"]:.6f}, max={row_audit.loc[("count", "row_normalized"), "row_sum_max"]:.6f}',
    f'- capacity raw row sums: min={row_audit.loc[("capacity_mw", "raw"), "row_sum_min"]:.6f}, max={row_audit.loc[("capacity_mw", "raw"), "row_sum_max"]:.6f}',
    f'- capacity row-normalized row sums: min={row_audit.loc[("capacity_mw", "row_normalized"), "row_sum_min"]:.6f}, max={row_audit.loc[("capacity_mw", "row_normalized"), "row_sum_max"]:.6f}',
    '',
    '## Interpretation rule',
    '- Raw matrices are diagnostic only.',
    '- Row-normalized matrices are the approved analytical inputs for later projection-based camp analysis.',
    '',
    '## Go/stop outcome',
    '- PASS: row normalization materially reduces head-country dominance for both count and capacity under the frozen projection rules.',
])
(REPORTS / 'p2_normalization_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

display(concentration_diagnostics)
display(Markdown('**Saved outputs**'))
display(pd.DataFrame({'path': [
    str(TABLES / 'p2_country_mode_count_raw.csv'),
    str(TABLES / 'p2_country_mode_capacity_raw.csv'),
    str(TABLES / 'p2_country_mode_count_row_normalized.csv'),
    str(TABLES / 'p2_country_mode_capacity_row_normalized.csv'),
    str(TABLES / 'p2_concentration_diagnostics.csv'),
    str(REPORTS / 'p2_normalization_report.md'),
    str(RUNTIME / 'p2_normalization_manifest.json'),
]}))
